# 03. CNN 기초

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_03_CNN_%EA%B8%B0%EC%B4%88.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## PyTorch로 CNN 구현하기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 입력: (B, 1, 28, 28) MNIST 가정
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)         # 2x2 최대 풀링
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))    # (B, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))    # (B, 32, 7, 7)
        x = x.view(x.size(0), -1)               # (B, 32*7*7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## SAR(합성개구레이더) 영상 표적 식별


In [ ]:
import torch
import torch.nn as nn

class SARClassifier(nn.Module):
    """SAR 영상 표적 분류기 (1채널 입력)"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                       # 128 -> 64
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                       # 64 -> 32
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),          # 임의 입력 크기 대응
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 입력: SAR 패치 (B, 1, 128, 128), log 스케일 정규화 가정
model = SARClassifier(num_classes=10)
print(model(torch.zeros(4, 1, 128, 128)).shape)

**실행 결과 예시**

```
torch.Size([4, 10])
```


## 실습: CIFAR-10으로 CNN 학습

### 🔬 실습 — CIFAR-10 CNN 학습 파이프라인

① **데이터 로딩과 증강** — 훈련 세트에만 증강을 적용한다.


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),        # 패딩 후 무작위 잘라내기
    transforms.RandomHorizontalFlip(p=0.5),      # 50% 확률 좌우 반전
    transforms.ColorJitter(0.2, 0.2, 0.1),       # 밝기/대비/채도 변화
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),   # 채널별 평균
                         (0.2023, 0.1994, 0.2010)),  # 채널별 표준편차
])
test_transform = transforms.Compose([            # 테스트: 정규화만
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
train_set = datasets.CIFAR10("./data", train=True,  download=True, transform=train_transform)
test_set  = datasets.CIFAR10("./data", train=False, download=True, transform=test_transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

② **모델 정의** — CIFAR-10은 3채널, 풀링을 2번 거치면 \(88\).


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)           # 합성곱 뒤 배치 정규화(2장)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)   # 32->16->8
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # Conv-BN-ReLU
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

③ **학습 설정** — 손실함수, 옵티마이저, 코사인 학습률 스케줄.


In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CIFAR10CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

④ **학습/평가 루프** — 매 에포크마다 학습 후 테스트 정확도를 출력한다.


In [ ]:
def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

for epoch in range(1, 31):
    model.train()                                   # ---- 학습 ----
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()                                    # ---- 평가 ----
    test_acc = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            test_acc += accuracy(model(x), y)
    test_acc /= len(test_loader)
    scheduler.step()
    print(f"Ep {epoch:02d} | test acc {test_acc:.3f}")

### 🔬 실습 — 혼동행렬로 CIFAR-10 모델 진단하기

① **테스트 셋 전체의 예측 수집과 혼동행렬 계산**


In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix

classes = ['plane','car','bird','cat','deer',
           'dog','frog','horse','ship','truck']

model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(1).cpu()
        preds.append(p); labels.append(y)
preds  = torch.cat(preds).numpy()
labels = torch.cat(labels).numpy()

cm = confusion_matrix(labels, preds)          # (10, 10) 행렬
per_class_acc = cm.diagonal() / cm.sum(1)     # 클래스별 재현율
for c, a in zip(classes, per_class_acc):
    print(f"{c:6s}: {a:.3f}")

**실행 결과 예시**

```
plane : 0.86  car : 0.92  bird : 0.74  cat : 0.65  deer : 0.81 \\
dog : 0.73  frog : 0.88  horse : 0.86  ship : 0.90  truck : 0.89
```


② **가장 헷갈리는 쌍 찾기**: 대각선을 지운 뒤 가장 큰 칸을 뽑는다.


In [ ]:
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f"최다 혼동: 실제 '{classes[i]}' -> 예측 '{classes[j]}' ({cm_off[i,j]}건)")

**실행 결과 예시**

```
최다 혼동: 실제 'cat' -> 예측 'dog' (112건)
```


## C. 구현 및 실험

`CIFAR10CNN`의 마지막 `x.view()`+`fc1`을 `nn.AdaptiveAvgPool2d((1,1))`+`nn.Linear(64, num_classes)`로 교체하고, 두 방식의 파라미터 수를 비교하라. 입력 해상도를 \(3232\)에서 \(4848\)로 바꿔도 동작하는지 각각 확인하라. 아래 잔차 블록의 `TODO`를 완성하라.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        # in_ch != out_ch 또는 stride != 1 이면 shortcut을 1x1 Conv로 맞춘다
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch))
        else:
            self.shortcut = nn.Identity()   # 모양이 같으면 그대로 통과

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))   # conv1 -> bn1 -> relu
        out = self.bn2(self.conv2(out))         # conv2 -> bn2
        out = out + self.shortcut(x)            # 잔차 연결
        return F.relu(out)

# 동작 확인: 채널이 바뀌는 블록(shortcut 필요)
print(ResidualBlock(32, 64, stride=2)(torch.zeros(1, 32, 16, 16)).shape)

`shortcut`은 입력과 출력의 모양이 다를 때(채널 수가 바뀌거나 stride로 공간이 줄 때) 맞춰 주는 \(11\) 합성곱이다.


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
